# 🎙️ TTS Text Optimizer for DesiVocal.com — English-India Edition

**Prepare public-domain books in Indian English for natural, professional audiobook TTS generation.**

This notebook will:
1. **Setup Ollama** — Install and run Ollama locally in Colab.
2. **Download Model** — Pull `gemma3:27b` (recommended) or another high-quality LLM.
3. **Optimize Text** — Upload your `.txt` file; the AI will reformat it for DesiVocal.com's TTS engine.
4. **Download Result** — Save the polished text, ready to paste into DesiVocal.com.

### 🇮🇳 What this optimizer does for English-India audiobooks:
- Converts American/British spellings and idioms → Indian English equivalents
- Expands abbreviations, acronyms, titles (Mr., Dr., govt., etc.)
- Formats dates, times, currencies, and numbers for natural speech
- Expands emails and URLs into spoken form
- Adds proper punctuation pauses for pacing
- Tags dialogue speakers clearly for single-voice TTS
- Handles the DesiVocal "10" bug and other engine quirks

✨ **Supports large texts with automatic chunking and progress tracking!**

## 📦 Step 1: Install & Setup Ollama
Run this cell to install Ollama and start the server in the background.

In [ ]:
# Install required packages
!pip install -q ollama requests ipywidgets

import subprocess
import time
import os
import sys

print("🦙 Installing Ollama...")

# Install zstd (required for Ollama extraction)
!apt-get update -qq && apt-get install -y -qq zstd > /dev/null 2>&1

# Download and install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

print("\n🚀 Starting Ollama server in background...")

os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
subprocess.Popen(['/usr/local/bin/ollama', 'serve'],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

time.sleep(5)

try:
    import ollama
    ollama.list()
    print("✅ Ollama server is running and ready!")
except Exception as e:
    print(f"⚠️  Ollama server may not be ready yet. Error: {e}")
    print("   Wait a few seconds and run the next cell.")

## 📥 Step 2: Select & Download Model
`gemma3:27b` is recommended — excellent English understanding with culturally aware output.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import ollama

print("🦙 Ollama Model Selection")
print("=" * 40)

OLLAMA_MODELS = {
    "gemma3:27b  ✅ Recommended — Best for Indian English": "gemma3:27b",
    "qwen2.5:14b — High quality, faster": "qwen2.5:14b",
    "qwen3:14b   — Thinking model, thorough": "qwen3:14b",
    "llama3.1:8b — Fastest, lower quality": "llama3.1:8b",
}

model_dropdown = widgets.Dropdown(
    options=list(OLLAMA_MODELS.keys()),
    value="gemma3:27b  ✅ Recommended — Best for Indian English",
    description='Model:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

display(model_dropdown)
print("\nSelect a model above, then run the next cell to download it.")

In [ ]:
# Pull the selected model
selected_model_name = OLLAMA_MODELS[model_dropdown.value]
print(f"📥 Pulling model: {selected_model_name}...")
print("   This may take several minutes depending on your internet speed.")

try:
    current_digest = ''
    for progress in ollama.pull(selected_model_name, stream=True):
        digest = progress.get('digest', '')
        if digest != current_digest and current_digest:
            print()
        current_digest = digest
        status = progress.get('status', '')
        if 'completed' in progress and 'total' in progress:
            completed = progress['completed']
            total = progress['total']
            pct = (completed / total * 100) if total > 0 else 0
            print(f"\r   {status}: {pct:.1f}%", end='', flush=True)
        else:
            print(f"\r   {status}", end='', flush=True)
    print(f"\n\n✅ Model '{selected_model_name}' is ready!")
except Exception as e:
    print(f"\n❌ Error pulling model: {e}")

## 🧠 Step 3: Define the English-India TTS Optimizer

This class contains the full prompt engineering for DesiVocal.com's English TTS engine, covering:
- Indian English vocabulary & phrasing
- Abbreviations, titles, full forms
- Dates, times, currencies, numbers
- Email/URL expansion
- Dialogue speaker tagging
- Pause punctuation for natural pacing
- Known DesiVocal engine quirks (the "10" bug, etc.)

In [ ]:
import requests
import json
import sys
import re
import time

class TTSOptimizerEnglishIndia:
    """Optimizes English-India text for DesiVocal.com TTS audiobook generation."""

    def __init__(self, model_name="gemma3:27b", chunk_size=1800, timeout=600):
        """
        Args:
            model_name : Ollama model to use.
            chunk_size : Max characters per chunk (1800 recommended for gemma3:27b).
            timeout    : Request timeout in seconds per chunk.
        """
        self.ollama_url = "http://localhost:11434/api/generate"
        self.model = model_name
        self.chunk_size = chunk_size
        self.timeout = timeout
        print(f"🤖 TTS Optimizer (English-India) initialized")
        print(f"   Model     : {self.model}")
        print(f"   Chunk size: {self.chunk_size} chars")
        print(f"   Timeout   : {self.timeout}s per chunk")

    # ─────────────────────────────────────────────
    # CHUNKING
    # ─────────────────────────────────────────────
    def chunk_text(self, text: str) -> list:
        """Split text into sentence-boundary-aligned chunks."""
        if len(text) <= self.chunk_size:
            return [text]

        chunks = []
        current_chunk = ""
        # Split on sentence-ending punctuation followed by whitespace
        sentences = re.split(r'([.!?]\s+)', text)

        for i in range(0, len(sentences), 2):
            sentence = sentences[i]
            separator = sentences[i + 1] if i + 1 < len(sentences) else ""
            if (len(current_chunk) + len(sentence) + len(separator) > self.chunk_size
                    and current_chunk):
                chunks.append(current_chunk.strip())
                current_chunk = sentence + separator
            else:
                current_chunk += sentence + separator

        if current_chunk.strip():
            chunks.append(current_chunk.strip())

        # Fallback: hard split if no sentence boundaries found
        if not chunks:
            chunks = [text[i:i + self.chunk_size]
                      for i in range(0, len(text), self.chunk_size)]

        print(f"\n📊 Split into {len(chunks)} chunks")
        for idx, c in enumerate(chunks, 1):
            print(f"   Chunk {idx}: {len(c)} characters")
        return chunks

    # ─────────────────────────────────────────────
    # PROMPT
    # ─────────────────────────────────────────────
    def get_optimization_prompt(self, text: str) -> str:
        prompt = f"""You are an expert text formatter for DesiVocal.com's English TTS engine. Your job is to reformat English-India public-domain book text so it sounds natural and clear when read aloud by a single AI voice.

═══════════════════════════════════════════════════════════════
RULE 1 — WORD PRESERVATION (CRITICAL)
═══════════════════════════════════════════════════════════════

Every meaningful word from the input MUST appear in the output.

ALLOWED changes:
- Expand abbreviations and acronyms
- Rewrite numbers, dates, times, currencies in spoken form
- Expand emails and URLs
- Add or change punctuation for pacing
- Add speaker tags to dialogue lines
- Remove dialogue attribution phrases ("he said", "she asked")
- Convert American/British spellings to Indian English equivalents

FORBIDDEN:
- Dropping words or sentences
- Adding your own explanations, commentary, or metadata
- Translating into another language
- Summarising or paraphrasing meaning

═══════════════════════════════════════════════════════════════
RULE 2 — INDIAN ENGLISH VOCABULARY
═══════════════════════════════════════════════════════════════

This audiobook is for Indian listeners. Use Indian English phrasing where natural.

COMMON REPLACEMENTS (apply only where the original word is present):
apartment / flat        → flat
elevator               → lift
sidewalk / pavement    → footpath
cell phone / cellphone → mobile
gas (fuel)             → petrol
attorney / lawyer      → advocate
vacation               → holiday
math                   → maths
candy / sweets         → sweets
truck                  → lorry (if heavy goods vehicle context)

SPELLING — use Indian English / Commonwealth spellings:
color → colour
honor → honour
center → centre
program → programme (unless software)
license (noun) → licence
realize → realise
analyze → analyse
organize → organise
traveled → travelled
modeling → modelling

DO NOT force Indian English where the original phrasing is specific to the story setting.

═══════════════════════════════════════════════════════════════
RULE 3 — DIALOGUE SPEAKER IDENTIFICATION
═══════════════════════════════════════════════════════════════

DesiVocal uses ONE VOICE. Mark every speaker clearly.

IDENTIFY SPEAKERS FROM CONTEXT:
"Holmes said" → speaker is Holmes
"Watson asked" → speaker is Watson
"I replied" → speaker is the narrator (I)
"he said" → look back 2–3 sentences to identify who "he" is

SPEAKER PRIORITY ORDER:
1. Named characters (Holmes, Watson, Raj, Priya)
2. Roles/titles (Inspector, Doctor, King, Colonel)
3. Resolved pronouns (track who "he"/"she"/"they" refers to)
4. LAST RESORT: Speaker1, Speaker2 (only if no context available)

ALTERNATING DIALOGUE:
If two characters alternate: first quote → Character A, second → Character B, etc.
Continue alternating unless context says otherwise.

REMOVE ATTRIBUTION WORDS:
WRONG: Holmes said: 'This is important.'
RIGHT:  Holmes: 'This is important.'

Each speaker on a NEW LINE with a blank line before it.

EXAMPLES:

INPUT:
Holmes said, "This is important." Watson asked, "Why?" "Because," Holmes explained, "the evidence is clear."

OUTPUT:
Holmes: 'This is important.'

Watson: "Why?"

Holmes: 'Because, the evidence is clear.'

---

INPUT:
"Come along," said Raj to Priya. "Now?" she asked. "Yes," he replied.

OUTPUT:
Raj: «Come along.»

Priya: *Now?*

Raj: «Yes.»

═══════════════════════════════════════════════════════════════
RULE 4 — DIFFERENT PUNCTUATION PER SPEAKER
═══════════════════════════════════════════════════════════════

Use DIFFERENT punctuation for EACH character to help listeners tell them apart.

ASSIGNMENT:
Main protagonist        → 'single quotes'
Secondary character     → "double quotes"
Authority/antagonist    → *asterisks*
Additional characters   → «guillemets»

CONSISTENCY: Each character keeps the same punctuation throughout the ENTIRE text.

FORMAT:
CharacterName: [mark]dialogue[mark]

Holmes: 'This is most irregular.'
Watson: "I quite agree."
Lestrade: *You are both wrong.*
Irene: «I beg to differ.»

═══════════════════════════════════════════════════════════════
RULE 5 — TECHNICAL FORMATTING FOR DESIVOCAL TTS
═══════════════════════════════════════════════════════════════

5A. THE "10" BUG (CRITICAL — DesiVocal cannot read "10" or "ten" correctly)
ALWAYS replace any standalone "10" with the word "ten":
10 men      → ten men
Chapter 10  → Chapter ten
10 AM       → ten AM
8–10 years  → 8 to ten years
100 is fine — only standalone 10 needs replacement.

5B. ROMAN NUMERALS → Arabic numbers
Chapter I → Chapter 1
Part II   → Part 2
King George VI → King George 6
I, II, III, IV, V, VI, VII, VIII, IX, X → 1, 2, 3, 4, 5, 6, 7, 8, 9, ten
XI, XII, ... → 11, 12, ...

5C. NUMBERS — remove commas, write out clearly
50,000      → 50000
1,00,000    → 100000
1,50,000    → 150000
3.14        → 3 point 14  (only in non-currency decimal contexts)

5D. DATES — full spoken form
15/03/1947  → 15th March 1947
03-07-2001  → 3rd July 2001
March 3rd, 1922 → 3rd March 1922  (reorder to Indian style: day month year)
Jan. 5      → 5th January

5E. YEARS — just say the year clearly (no prefix needed in English)
1988 → nineteen eighty-eight  (if in a sentence like "born in 1988")
OR keep as 1988 — DesiVocal handles 4-digit years well in English.
NEVER write "in the year 1988" unless the original says so.

5F. TIME — write in words
3:30 PM  → half past three in the afternoon
10:00 AM → ten o'clock in the morning
6:15     → quarter past six
9:45 PM  → quarter to ten in the evening
Midnight → midnight

5G. CURRENCY — Indian spoken form
Rs. 500     → five hundred rupees
₹ 1,50,000  → one lakh fifty thousand rupees
$50         → fifty dollars
£20         → twenty pounds
€30         → thirty euros
Re. 1       → one rupee

5H. PERCENTAGES
50%   → fifty percent
2.5%  → two point five percent

5I. ABBREVIATIONS & TITLES — always expand
Mr.     → Mister
Mrs.    → Missus
Ms.     → Miss  (or Miz if context implies divorced/unmarried adult)
Dr.     → Doctor
Prof.   → Professor
Capt.   → Captain
Col.    → Colonel
Lt.     → Lieutenant
Sgt.    → Sergeant
Insp.   → Inspector
Sr.     → Senior
Jr.     → Junior
govt.   → government
dept.   → department
admin.  → administration
approx. → approximately
viz.    → namely
i.e.    → that is
e.g.    → for example
etc.    → and so on
vs.     → versus
St.     → Saint (if saint context) OR Street (if address context)
Ave.    → Avenue
Rd.     → Road
No.     → Number
Vol.    → Volume
ed.     → edition
pp.     → pages
Ch.     → Chapter
Fig.    → Figure
Ref.    → Reference

5J. ACRONYMS — remove all internal periods, keep as spoken letters or words
U.S.A.   → USA
U.K.     → UK
N.A.S.A. → NASA
B.B.C.   → BBC
I.A.S.   → IAS
I.P.S.   → IPS
R.B.I.   → RBI
I.I.T.   → IIT
C.B.I.   → CBI
P.M.     → Prime Minister  (if political context)
C.M.     → Chief Minister
M.P.     → Member of Parliament (or Madhya Pradesh — use context)
M.L.A.   → MLA
I.O.U.   → I owe you
A.M.     → in the morning
P.M.     → in the evening  (if time context)

5K. EMAILS & URLS — expand to spoken form
@             → at the rate
. (in URL/email) → dot
/ (in URL)    → slash
hr@company.com   → h r at the rate company dot com
www.example.in   → www dot example dot in
https://abc.org  → h t t p s colon slash slash a b c dot org

5L. SYMBOLS — expand to full words
&   → and
+   → plus
–   → to  (in ranges like 5–8)
-   → to  (in ranges) OR hyphen (in compound words — remove it)
×   → times
÷   → divided by
=   → equals
°F  → degrees Fahrenheit
°C  → degrees Celsius
km  → kilometres
kg  → kilograms
cm  → centimetres
mm  → millimetres
m   → metres  (if unit of measurement)
km² → square kilometres
#   → number  (if before a number) OR hash (if hashtag)
§   → section
©   → copyright
™   → trademark
®   → registered trademark

5M. RANGES — use "to" with a space
5–8       → 5 to 8
10–15     → ten to 15
1990–2000 → 1990 to 2000
pp. 5–8   → pages 5 to 8

5N. COMPOUND WORDS — remove hyphens, write as two words
cross-check    → cross check
well-known     → well known
long-term      → long term
short-sighted  → short sighted
(Exception: keep hyphens in proper names like "Gupta-Sharma" if they appear)

5O. CHAPTER/SECTION HEADERS — make TTS-friendly
Chapter I     → Chapter 1.
Chapter X     → Chapter ten.
Part III      → Part 3.
CHAPTER ONE   → Chapter 1.
Section 2.1   → Section 2 point 1.

5P. NAMES — Indian name pronunciation helpers
If a Western name appears that may be mispronounced, keep as-is (TTS handles common names).
If an Indian name appears with unusual spelling, do not alter it.

═══════════════════════════════════════════════════════════════
RULE 6 — PAUSE PUNCTUATION FOR NATURAL PACING
═══════════════════════════════════════════════════════════════

Use punctuation to guide the TTS engine's rhythm:
, = short pause
| = medium pause (scene or context shift)
. = long pause (end of thought)
,, = extended pause (dramatic moment)
... = suspense / trailing thought
!! = excitement or alarm
?? = confusion or disbelief

Add pauses at:
- Chapter transitions
- Scene changes (new location, time jump)
- After a dramatic reveal
- Before a list of items
- Between a question and its answer

EXAMPLE:
INPUT: The door opened. A man stood there. It was Holmes.
OUTPUT: The door opened. | A man stood there.,, | It was Holmes.

═══════════════════════════════════════════════════════════════
COMPREHENSIVE EXAMPLES
═══════════════════════════════════════════════════════════════

EXAMPLE 1 — Chapter header + narration + dates + numbers

INPUT:
Chapter I
It was March 3rd, 1922. Mr. Sherlock Holmes had solved approx. 50 cases that year. Dr. Watson, his companion, noted that Holmes's fee was Rs. 500 per case — a sum totaling Rs. 25,000.

OUTPUT:
Chapter 1.

It was 3rd March 1922. Mister Sherlock Holmes had solved approximately 50 cases that year. Doctor Watson, his companion, noted that Holmes's fee was five hundred rupees per case, a sum totalling twenty-five thousand rupees.

---

EXAMPLE 2 — Dialogue with speaker tagging

INPUT:
"Come at once if convenient," Holmes said. "Is it not convenient?" Watson replied. "Preferably at once," said Holmes.

OUTPUT:
Holmes: 'Come at once if convenient.'

Watson: "Is it not convenient?"

Holmes: 'Preferably at once.'

---

EXAMPLE 3 — Abbreviations, email, URL, time

INPUT:
Insp. Lestrade contacted Dr. Watson at watson@221b.co.uk at 10:30 PM on Jan. 5. See www.scotland-yard.gov.uk for more info.

OUTPUT:
Inspector Lestrade contacted Doctor Watson at watson at the rate 221b dot co dot uk at half past ten in the evening on 5th January. See www dot scotland yard dot gov dot uk for more information.

---

EXAMPLE 4 — Numbers, ranges, symbols, the "10" bug

INPUT:
The temperature was 10°C. He had traveled 10 km in 8–10 minutes. He was 6'2" tall and weighed 85 kg. The U.S.A. report covered pp. 5–10.

OUTPUT:
The temperature was ten degrees Celsius. He had travelled ten kilometres in 8 to ten minutes. He was 6 foot 2 inches tall and weighed 85 kilograms. The USA report covered pages 5 to ten.

---

EXAMPLE 5 — Pronoun resolution in dialogue

INPUT:
Holmes was in the room. Watson entered and asked, "Are you ready?" He replied, "Almost." "When?" Watson asked.

OUTPUT:
Holmes was in the room. Watson entered.

Watson: "Are you ready?"

Holmes: 'Almost.'

Watson: "When?"

---

EXAMPLE 6 — Currency, percentage, chapter header

INPUT:
Chapter X
The govt. budget was Rs. 1,50,000. Approx. 30% was spent on admin. costs. The C.B.I. investigated.

OUTPUT:
Chapter ten.

The government budget was one lakh fifty thousand rupees. Approximately 30 percent was spent on administration costs. The CBI investigated.

═══════════════════════════════════════════════════════════════
PROCESSING WORKFLOW
═══════════════════════════════════════════════════════════════

STEP 1: Read entire chunk; identify all characters and narrative voice.
STEP 2: Assign punctuation markers to each character (consistent):
  - Main protagonist → 'single quotes'
  - Secondary character → "double quotes"
  - Authority/antagonist → *asterisks*
  - Others → «guillemets»
STEP 3: Format dialogue with speaker tags, remove attribution phrases.
STEP 4: Apply all technical formatting (numbers, dates, abbr., symbols).
STEP 5: Apply Indian English vocabulary and spelling where appropriate.
STEP 6: Add pause punctuation for natural TTS pacing.
STEP 7: Verify the checklist below.

VERIFICATION CHECKLIST:
[ ] All standalone "10" replaced with "ten"
[ ] All Roman numerals → Arabic numbers
[ ] All abbreviations expanded (Mr., Dr., govt., etc.)
[ ] All acronyms have periods removed (U.S.A. → USA)
[ ] All dates in day-month-year spoken form
[ ] All times written in words
[ ] All currencies in spoken form (rupees, dollars, etc.)
[ ] All percentages written as "X percent"
[ ] All email/URL symbols expanded (@ → at the rate, . → dot)
[ ] All symbols expanded (&, ×, °C, km, etc.)
[ ] All number ranges use "to" (5–8 → 5 to 8)
[ ] All compound word hyphens removed
[ ] Indian English spellings applied (colour, centre, organise, etc.)
[ ] All speakers identified from context, tagged on new lines
[ ] Each character has consistent punctuation style
[ ] Attribution words removed (said, asked, replied, etc.)
[ ] Pause punctuation added at scene breaks and dramatic moments
[ ] No words dropped; no added commentary

═══════════════════════════════════════════════════════════════
OUTPUT INSTRUCTIONS
═══════════════════════════════════════════════════════════════

Return ONLY the formatted text.
NO explanations.
NO bullet points.
NO metadata.
NO markdown headers.
Just clean formatted text ready for DesiVocal.com TTS.

═══════════════════════════════════════════════════════════════
INPUT TEXT:
{text}
"""
        return prompt

    # ─────────────────────────────────────────────
    # SINGLE CHUNK PROCESSING
    # ─────────────────────────────────────────────
    def optimize_chunk(self, chunk: str, retry_count: int = 3) -> str:
        """Send one chunk to Ollama and return the optimized text."""
        prompt = self.get_optimization_prompt(chunk)
        payload = {
            "model": self.model,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0.3,   # Lower = more deterministic/consistent
                "top_p": 0.9,
                "num_predict": -1
            }
        }

        for attempt in range(retry_count):
            try:
                response = requests.post(
                    self.ollama_url, json=payload, timeout=self.timeout)
                response.raise_for_status()
                result = response.json()
                optimized = result.get("response", "").strip()
                return self._clean_output(optimized)
            except requests.exceptions.Timeout:
                if attempt < retry_count - 1:
                    wait = (attempt + 1) * 10
                    print(f"\n⚠️  Timeout on attempt {attempt+1}/{retry_count}. Retrying in {wait}s...")
                    time.sleep(wait)
                else:
                    print(f"\n❌ Failed after {retry_count} attempts (timeout)")
                    raise
            except Exception as e:
                if attempt < retry_count - 1:
                    wait = (attempt + 1) * 5
                    print(f"\n⚠️  Error on attempt {attempt+1}/{retry_count}: {e}")
                    print(f"   Retrying in {wait}s...")
                    time.sleep(wait)
                else:
                    print(f"\n❌ Failed after {retry_count} attempts: {e}")
                    raise
        return None

    # ─────────────────────────────────────────────
    # FULL TEXT PROCESSING
    # ─────────────────────────────────────────────
    def optimize(self, text: str) -> str:
        """Optimize full text with automatic chunking."""
        chunks = self.chunk_text(text)

        if len(chunks) == 1:
            print(f"\n📤 Processing single chunk ({len(text)} chars)...")
            return self.optimize_chunk(chunks[0])

        print(f"\n🔄 Processing {len(chunks)} chunks...")
        optimized_chunks = []

        for idx, chunk in enumerate(chunks, 1):
            print(f"\n📤 Processing chunk {idx}/{len(chunks)} ({len(chunk)} chars)...")
            try:
                optimized = self.optimize_chunk(chunk)
                if optimized:
                    optimized_chunks.append(optimized)
                    print(f"✅ Chunk {idx}/{len(chunks)} complete!")
                else:
                    print(f"⚠️  Chunk {idx}/{len(chunks)} returned empty — using original")
                    optimized_chunks.append(chunk)
            except Exception as e:
                print(f"❌ Error on chunk {idx}: {e}")
                print("   Using original chunk text.")
                optimized_chunks.append(chunk)

        final_text = "\n\n".join(optimized_chunks)
        print(f"\n✅ All chunks done! Total output: {len(final_text)} characters")
        return final_text

    # ─────────────────────────────────────────────
    # CLEANUP
    # ─────────────────────────────────────────────
    def _clean_output(self, text: str) -> str:
        """Strip markdown and formatting artifacts from model output."""
        text = text.replace("```", "").replace("**", "")
        lines = [
            line.strip() for line in text.split('\n')
            if line.strip()
            and not line.strip().startswith('#')
            and not line.strip().startswith('OUTPUT')
            and not line.strip().startswith('INPUT')
        ]
        return '\n'.join(lines).strip()


print("✅ TTSOptimizerEnglishIndia class loaded!")

## 📝 Step 4: Configure & Upload Your Text File

Set your chunk size (1800 chars works well for `gemma3:27b`) and upload your `.txt` book file.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from google.colab import files

print("⚙️  Configuration")
print("=" * 40)

chunk_slider = widgets.IntSlider(
    value=1800,
    min=500,
    max=5000,
    step=100,
    description='Chunk Size:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

display(chunk_slider)
print("\n💡 Recommended: 1800 for gemma3:27b, 2500 for qwen2.5:14b")
print("\n📂 Now uploading your text file...")

uploaded = files.upload()

# Read the uploaded file
uploaded_filename = list(uploaded.keys())[0]
raw_text = uploaded[uploaded_filename].decode('utf-8')

print(f"\n✅ File uploaded: {uploaded_filename}")
print(f"   Total characters: {len(raw_text):,}")
print(f"   Total words     : {len(raw_text.split()):,}")
print(f"\n📖 Preview (first 500 chars):")
print("-" * 50)
print(raw_text[:500])
print("-" * 50)

## 🚀 Step 5: Run Optimization

This will process your book chunk by chunk. Larger books may take 15–60+ minutes depending on the model and your Colab GPU.

In [ ]:
import time

# Initialise the optimizer
optimizer = TTSOptimizerEnglishIndia(
    model_name=selected_model_name,
    chunk_size=chunk_slider.value,
    timeout=600
)

print(f"\n🎬 Starting optimization of '{uploaded_filename}'...")
print(f"   Input length: {len(raw_text):,} characters")
start_time = time.time()

optimized_text = optimizer.optimize(raw_text)

elapsed = time.time() - start_time
minutes, seconds = divmod(int(elapsed), 60)

print(f"\n{'='*50}")
print(f"✅ OPTIMIZATION COMPLETE")
print(f"   Time taken     : {minutes}m {seconds}s")
print(f"   Input length   : {len(raw_text):,} characters")
print(f"   Output length  : {len(optimized_text):,} characters")
print(f"   Change ratio   : {len(optimized_text)/len(raw_text):.2f}x")
print(f"{'='*50}")

print("\n📖 Preview of optimized output (first 1000 chars):")
print("-" * 50)
print(optimized_text[:1000])
print("-" * 50)

## 💾 Step 6: Save & Download Result

Download the optimized `.txt` file — ready to paste into DesiVocal.com!

In [ ]:
from google.colab import files

# Build output filename
base_name = uploaded_filename.replace('.txt', '')
output_filename = f"{base_name}_TTS_ready.txt"

# Save to file
with open(output_filename, 'w', encoding='utf-8') as f:
    f.write(optimized_text)

print(f"💾 Saved: {output_filename}")
print(f"   Size: {len(optimized_text):,} characters")
print("\n📥 Downloading...")

files.download(output_filename)

print("\n✅ Done! Your file is downloading.")
print("\n🎙️  Next steps:")
print("   1. Go to https://www.desivocal.com/")
print("   2. Paste your optimized text into the TTS editor")
print("   3. Select an English-India voice")
print("   4. Generate & export your audiobook chapter")
print("   5. Upload to YouTube (full audiobook) or Instagram (snippets)")

## 🔧 Step 7 (Optional): Quick Test on Sample Text

Test the optimizer on a short snippet before processing a full book.

In [ ]:
# ── Edit the sample text below to test your own passage ──
sample_text = """
Chapter I

It was Jan. 15, 1920, at 10:30 AM. Mr. Sherlock Holmes & Dr. Watson were seated in the flat at 221B Baker St. The temperature outside was 10°C. Holmes had solved approx. 10 cases that month — a record.

"Come at once," Holmes said. "Is it not possible?" Watson asked. "Preferably at once," said Holmes. "I.e., now."

Watson reached for his coat. He had traveled 10 km that morning. His fee was Rs. 500 per visit — totaling Rs. 5,000 for the month. The U.K. newspaper lay on the table. See www.baker-street.co.uk for the full report.
""".strip()

print("🧪 Running quick test...")
print("=" * 50)

test_optimizer = TTSOptimizerEnglishIndia(
    model_name=selected_model_name,
    chunk_size=2000,
    timeout=300
)

result = test_optimizer.optimize(sample_text)

print("\n📥 INPUT:")
print("-" * 40)
print(sample_text)
print("\n📤 OUTPUT:")
print("-" * 40)
print(result)

## 💡 Tips for Best Results

### Model Choice
| Model | Speed | Quality | Best For |
|---|---|---|---|
| `gemma3:27b` | Slow | Excellent | Full books, nuanced dialogue |
| `qwen2.5:14b` | Medium | Very good | Large chapters |
| `qwen3:14b` | Medium | Very good | Complex formatting |
| `llama3.1:8b` | Fast | Good | Quick tests |

### Chunk Size
- **1500–1800 chars** → Best quality (model has full context)
- **2000–2500 chars** → Good balance of speed and quality
- **3000+ chars** → May lose consistency at chunk boundaries

### Book Preparation
- Remove page numbers and headers/footers before uploading
- Split very long books into chapters and process one chapter at a time
- Use UTF-8 encoding for your `.txt` file
- Keep footnotes/endnotes separate; they don't work well in TTS

### DesiVocal.com
- Choose an Indian English voice (e.g., Aditi, Raveena)
- Set speech rate to 0.9x for audiobook feel
- Process max ~5000 chars per TTS job; split if needed
- Export as MP3 for YouTube and Instagram

### YouTube / Instagram Workflow
- **YouTube**: Full chapters (10–60 min), add a chapter book cover image
- **Instagram Reels**: 60–90 second snippets of exciting moments, add subtitles